# 04 · Active Learning Simulation

Simulate the active-learning loop (**RQ3**): start from a small seed, query informative PRs with different strategies, and watch the learning curve. The ground-truth labels act as the human oracle.

- **Inputs:** `data/sample/sample_prs.csv`
- **Outputs:** Learning curves comparing query strategies.

> ⚠️ **Sample vs. real data.** This notebook runs on the committed 10-row synthetic sample so the toolchain works without PRismBench. The sample has singleton classes, so metrics here are *illustrative only*. Each `TODO` marks where the real dataset in `data/raw/` plugs in.

In [ ]:
# --- Standard setup: locate project root, add src/ to path, load helpers ---
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    """Walk upwards until we find the repo root (has pyproject.toml + src/pr_risk)."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "pr_risk").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 50)
SAMPLE_CSV = PROJECT_ROOT / "data" / "sample" / "sample_prs.csv"
print("Project root :", PROJECT_ROOT)
print("Sample CSV   :", SAMPLE_CSV.name, "| exists:", SAMPLE_CSV.exists())

In [ ]:
import matplotlib.pyplot as plt

try:
    import seaborn as sns

    sns.set_theme(style="whitegrid")
    HAS_SNS = True
except ImportError:  # seaborn is optional; matplotlib is enough
    HAS_SNS = False
print("seaborn available:", HAS_SNS)

## 1. Data & features
Use metadata features and the binary target.

In [ ]:
from pr_risk.data.load_data import load_csv
from pr_risk.features.metadata_features import create_metadata_features

df = load_csv(SAMPLE_CSV)
df = df[df["is_risky"].isin([0, 1])].copy().reset_index(drop=True)
X = create_metadata_features(df)
y = df["is_risky"].astype(int)
print("pool size:", len(X), "| features:", list(X.columns))

## 2. Seed / pool / test split
We deterministically put **one of each class** in the seed and the test set so every
model trains on 2 classes (avoids single-class errors on the tiny sample).

In [ ]:
zero_idx = y.index[y == 0].tolist()
one_idx = y.index[y == 1].tolist()

test_idx = [zero_idx[0], one_idx[0], one_idx[1]]
seed_idx = [zero_idx[1], one_idx[2]]
pool_idx = [i for i in y.index if i not in test_idx + seed_idx]
print("seed:", seed_idx, "| pool:", pool_idx, "| test:", test_idx)

## 3. One query step
Train on the seed, score the pool, and let a strategy pick the next PR to label.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

from pr_risk.active_learning import query_strategies as qs

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X.loc[seed_idx], y.loc[seed_idx])
probas = model.predict_proba(X.loc[pool_idx])

picks = {
    "least_confidence": qs.least_confidence_sampling(probas, 1),
    "margin": qs.margin_sampling(probas, 1),
    "entropy": qs.entropy_sampling(probas, 1),
}
for name, local in picks.items():
    chosen_pool_row = pool_idx[int(local[0])]
    print(f"{name:16s} -> pool position {int(local[0])} (df row {chosen_pool_row})")

## 4. Simulate full rounds across strategies
Add one labelled PR per round; retrain; track test accuracy vs. labelled count.

In [ ]:
from sklearn.metrics import accuracy_score


def run_strategy(strategy: str, seed, pool, test):
    labelled, pool = list(seed), list(pool)
    history = []
    while True:
        clf = RandomForestClassifier(n_estimators=50, random_state=42)
        clf.fit(X.loc[labelled], y.loc[labelled])
        acc = accuracy_score(y.loc[test], clf.predict(X.loc[test]))
        history.append((len(labelled), acc))
        if not pool:
            break
        p = clf.predict_proba(X.loc[pool])
        if p.shape[1] < 2:
            local = 0
        elif strategy == "random":
            local = int(qs.random_sampling(len(pool), 1)[0])
        elif strategy == "least_confidence":
            local = int(qs.least_confidence_sampling(p, 1)[0])
        elif strategy == "margin":
            local = int(qs.margin_sampling(p, 1)[0])
        elif strategy == "entropy":
            local = int(qs.entropy_sampling(p, 1)[0])
        labelled.append(pool.pop(local))
    return history


curves = {s: run_strategy(s, seed_idx, pool_idx, test_idx)
          for s in ["random", "least_confidence", "margin", "entropy"]}

plt.figure(figsize=(7, 4))
for name, hist in curves.items():
    xs, ys = zip(*hist, strict=True)
    plt.plot(xs, ys, marker="o", label=name)
plt.xlabel("labelled examples")
plt.ylabel("test accuracy")
plt.title("Active-learning curves (sample · illustrative)")
plt.legend()
plt.tight_layout()
plt.show()

## Notes & TODO (real data)
- `diversity_sampling` is a **placeholder** (random fallback) — implement clustering / embedding-distance selection.
- `active_learning_loop` is a package **placeholder**; this notebook runs the loop inline. Implement the orchestrator next.
- On real data, curves should show uncertainty strategies (margin/entropy/least-confidence) beating random for the same label budget.
- Wire the loop to the Streamlit labeller (`app/labelling_app`) and `pr_risk.annotation` for true human-in-the-loop labelling. Config: `configs/active_learning.yaml`.